# Hyperparameter Tuning — Random Forest & XGBoost

Optimize baseline models using `GridSearchCV` (Random Forest) and `RandomizedSearchCV` (XGBoost). Compare tuned vs baseline metrics.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '..'))
MODELS_DIR    = os.path.join(BASE_DIR, 'models')
PROCESSED_DIR = os.path.join(BASE_DIR, 'processed')
os.makedirs(MODELS_DIR, exist_ok=True)

X_train = joblib.load(os.path.join(PROCESSED_DIR, 'X_train.pkl'))
X_test  = joblib.load(os.path.join(PROCESSED_DIR, 'X_test.pkl'))
y_train = joblib.load(os.path.join(PROCESSED_DIR, 'y_train.pkl'))
y_test  = joblib.load(os.path.join(PROCESSED_DIR, 'y_test.pkl'))

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

## Helper — Evaluate Model

In [ ]:
def evaluate(model, label='Model'):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    metrics = dict(
        acc  = round(accuracy_score(y_test, y_pred), 4),
        prec = round(precision_score(y_test, y_pred), 4),
        rec  = round(recall_score(y_test, y_pred), 4),
        f1   = round(f1_score(y_test, y_pred), 4),
        auc  = round(roc_auc_score(y_test, y_prob), 4)
    )
    print(f"\n{'='*45}")
    print(f"  {label}")
    print(f"{'='*45}")
    for k, v in metrics.items():
        print(f"  {k.capitalize():10}: {v:.4f}")
    return metrics

## Task 1 — Tune Random Forest with GridSearchCV

`GridSearchCV` exhaustively tests every parameter combination. Scoring is set to `f1` — the primary business metric for churn detection.

In [ ]:
param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'max_features':      ['sqrt', 'log2']
}

grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5, scoring='f1', n_jobs=-1, verbose=1
)
grid_rf.fit(X_train, y_train)

print("\nBest Parameters (Random Forest):")
print(grid_rf.best_params_)
print(f"\nBest CV F1 Score: {grid_rf.best_score_:.4f}")

In [ ]:
# Retrain on full X_train with best params
best_rf = RandomForestClassifier(**grid_rf.best_params_, random_state=42)
best_rf.fit(X_train, y_train)
rf_tuned_metrics = evaluate(best_rf, 'Random Forest — Tuned')

# Save tuned model
rf_save_path = os.path.join(MODELS_DIR, 'random_forest_tuned.pkl')
joblib.dump(best_rf, rf_save_path)
print(f"\nModel saved → {rf_save_path}")

## Task 2 — Tune XGBoost with RandomizedSearchCV

`RandomizedSearchCV` samples 30 random combinations from the parameter space — efficient for large grids.

In [ ]:
param_dist = {
    'n_estimators':     [100, 200, 300, 400],
    'max_depth':        [3, 5, 7, 9],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'subsample':        [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

rand_xgb = RandomizedSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0),
    param_distributions=param_dist,
    n_iter=30, cv=5, scoring='f1',
    random_state=42, n_jobs=-1, verbose=1
)
rand_xgb.fit(X_train, y_train)

print("\nBest Parameters (XGBoost):")
print(rand_xgb.best_params_)
print(f"\nBest CV F1 Score: {rand_xgb.best_score_:.4f}")

In [ ]:
# Retrain on full X_train with best params
best_xgb = XGBClassifier(
    **rand_xgb.best_params_,
    eval_metric='logloss', random_state=42, verbosity=0
)
best_xgb.fit(X_train, y_train)
xgb_tuned_metrics = evaluate(best_xgb, 'XGBoost — Tuned')

# Save tuned model
xgb_save_path = os.path.join(MODELS_DIR, 'xgboost_tuned.pkl')
joblib.dump(best_xgb, xgb_save_path)
print(f"\nModel saved → {xgb_save_path}")

## Task 3 — Baseline vs Tuned Comparison Table

Load baseline metrics from Phase 5 CSV and compare against tuned results. Positive deltas highlighted green, negative red.

In [ ]:
baseline_df = pd.read_csv(os.path.join(MODELS_DIR, 'results_summary.csv'))

rf_b  = baseline_df[baseline_df['Model'] == 'random_forest'].iloc[0]
xgb_b = baseline_df[baseline_df['Model'] == 'xgboost'].iloc[0]

comparison_df = pd.DataFrame({
    'Model':           ['Random Forest', 'XGBoost'],
    'Baseline_F1':     [rf_b['F1'],     xgb_b['F1']],
    'Tuned_F1':        [rf_tuned_metrics['f1'],  xgb_tuned_metrics['f1']],
    'Baseline_Recall': [rf_b['Recall'], xgb_b['Recall']],
    'Tuned_Recall':    [rf_tuned_metrics['rec'], xgb_tuned_metrics['rec']],
})

comparison_df['Delta_F1']     = (comparison_df['Tuned_F1']     - comparison_df['Baseline_F1']).round(4)
comparison_df['Delta_Recall'] = (comparison_df['Tuned_Recall'] - comparison_df['Baseline_Recall']).round(4)

# Save comparison
comparison_df.to_csv(os.path.join(MODELS_DIR, 'tuning_comparison.csv'), index=False)

# Highlight positive/negative deltas
def highlight_deltas(val):
    if isinstance(val, float):
        if val > 0:   return 'background-color: #c6efce; color: #276221'
        elif val < 0: return 'background-color: #ffc7ce; color: #9c0006'
    return ''

styled = (comparison_df.style
    .map(highlight_deltas, subset=['Delta_F1', 'Delta_Recall'])
    .set_caption("Hyperparameter Tuning — Baseline vs Tuned")
    .format({
        'Baseline_F1': '{:.4f}', 'Tuned_F1': '{:.4f}',
        'Baseline_Recall': '{:.4f}', 'Tuned_Recall': '{:.4f}',
        'Delta_F1': '{:+.4f}', 'Delta_Recall': '{:+.4f}'
    }))

display(styled)